In [ ]:
import time
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse as sp

from HARK.ConsumptionSaving.ConsNewKeynesianModel import (
    NewKeynesianConsumerType,
    init_newkeynesian,
)
from HARK.distributions import Lognormal as LognormalDist
from HARK.utilities import jump_to_grid_2D

COLOR_MC = "tab:blue"
COLOR_TM = "tab:orange"
COLOR_HARM = "tab:green"

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "axes.labelsize": 13,
        "axes.titlesize": 15,
        "legend.fontsize": 13,
        "lines.linewidth": 2.5,
    }
)

BURNIN = 500
N_MC_BINS = 200
N_P_DISC = 50
MAX_P_FAC = 10.0

timings = {}


def correct_newborn_dist(agent, param_dict, n_p_disc=N_P_DISC):
    """Patch the TM newborn distribution to match MC's lognormal pLvl init.

    HARK hardcodes newborns at pLvl=1.0.  This subtracts the default
    newborn column and adds a lognormal-distributed replacement so that
    TM and MC solve the same economic model.
    """
    p_init = LognormalDist(
        mu=param_dict["pLogInitMean"], sigma=param_dict["pLogInitStd"]
    )
    p_init_d = p_init.discretize(n_p_disc)
    p_vals_init = p_init_d.atoms.flatten()
    p_prbs_init = p_init_d.pmv.flatten()

    shk_prbs = agent.IncShkDstn[0].pmv
    old_NBD = jump_to_grid_2D(
        np.ones_like(shk_prbs),
        np.ones_like(shk_prbs),
        shk_prbs,
        agent.dist_mGrid,
        agent.dist_pGrid,
    )
    new_NBD = jump_to_grid_2D(
        np.ones(n_p_disc),
        p_vals_init,
        p_prbs_init,
        agent.dist_mGrid,
        agent.dist_pGrid,
    )
    live_prob = agent.LivPrb[0]
    correction = (1.0 - live_prob) * (new_NBD - old_NBD)
    agent.tran_matrix += correction[:, np.newaxis]


def create_finite_horizon_agent(
    ss_agent, param_dict, T_cycle, shock_t, dx, orig_IncShkDstn
):
    """Create a finite-horizon agent for perfect-foresight transition paths.

    Returns a solved agent with time-varying Rfree that includes a one-period
    interest-rate deviation of size dx at period shock_t.
    """
    params = deepcopy(param_dict)
    params["T_cycle"] = T_cycle
    params["LivPrb"] = T_cycle * [ss_agent.LivPrb[0]]
    params["PermGroFac"] = T_cycle * [1.0]
    params["PermShkStd"] = T_cycle * [ss_agent.PermShkStd[0]]
    params["TranShkStd"] = T_cycle * [ss_agent.TranShkStd[0]]
    params["tax_rate"] = T_cycle * [ss_agent.tax_rate[0]]
    params["labor"] = T_cycle * [ss_agent.labor[0]]
    params["wage"] = T_cycle * [ss_agent.wage[0]]
    params["Rfree"] = T_cycle * [ss_agent.Rfree]
    params["DiscFac"] = T_cycle * [ss_agent.DiscFac]

    agent = NewKeynesianConsumerType(**params)
    agent.cycles = 1

    agent.del_from_time_inv("Rfree")
    agent.add_to_time_vary("Rfree")
    agent.del_from_time_inv("DiscFac")
    agent.add_to_time_vary("DiscFac")

    # Use the ORIGINAL income distribution — not the neutral-measure version
    agent.IncShkDstn = T_cycle * [orig_IncShkDstn]
    # Set the FULL terminal solution (not just cFunc_terminal_, which the
    # solver ignores — it reads solution_terminal instead)
    agent.solution_terminal = deepcopy(ss_agent.solution[0])

    R = ss_agent.Rfree[0]
    agent.Rfree = shock_t * [R] + [R + dx] + (T_cycle - shock_t - 1) * [R]

    return agent


def jump_to_grid_fast(m_vals, probs, dist_mGrid):
    """Distribute probability mass onto a grid, preserving means.

    Like HARK's jump_to_grid_1D but with a simpler interface.  Each value
    in m_vals has its probability split between the two nearest grid points
    using linear interpolation weights.
    """
    probGrid = np.zeros(len(dist_mGrid))
    mIndex = np.digitize(m_vals, dist_mGrid) - 1
    mIndex[m_vals <= dist_mGrid[0]] = -1
    mIndex[m_vals >= dist_mGrid[-1]] = len(dist_mGrid) - 1

    for i in range(len(m_vals)):
        if mIndex[i] == -1:
            mlowerIndex = 0
            mupperIndex = 0
            mlowerWeight = 1.0
            mupperWeight = 0.0
        elif mIndex[i] == len(dist_mGrid) - 1:
            mlowerIndex = -1
            mupperIndex = -1
            mlowerWeight = 1.0
            mupperWeight = 0.0
        else:
            mlowerIndex = mIndex[i]
            mupperIndex = mIndex[i] + 1
            mlowerWeight = (dist_mGrid[mupperIndex] - m_vals[i]) / (
                dist_mGrid[mupperIndex] - dist_mGrid[mlowerIndex]
            )
            mupperWeight = 1.0 - mlowerWeight

        probGrid[mlowerIndex] += probs[i] * mlowerWeight
        probGrid[mupperIndex] += probs[i] * mupperWeight

    return probGrid.flatten()

In [ ]:
# Start from HARK's NewKeynesian defaults (infinite horizon, cycles=0)
# and override with cstwMPC quarterly calibration (Carroll et al. 2017).
LivPrb_quarterly = 1.0 - 1.0 / 160.0  # Blanchard–Yaari, ≈ 40-year expected life

Dict = {
    **init_newkeynesian,
    # --- Preferences (cstwMPC β-Point) ---
    "CRRA": 1.01,  # near-log utility
    "Rfree": [1.01 / LivPrb_quarterly],  # mortality-adjusted quarterly rate
    "DiscFac": 0.9867,  # β-Point estimate
    "LivPrb": [LivPrb_quarterly],
    # --- Income process (Sabelhaus & Song 2010, via cstwMPC) ---
    "PermShkStd": [(0.01 * 4 / 11) ** 0.5],  # ≈ 0.0603
    "TranShkStd": [(0.01 * 4) ** 0.5],  # = 0.2
    "UnempPrb": 0.07,
    "IncUnemp": 0.15,
    "UnempPrbRet": 0.0005,
    # --- Simulation ---
    "AgentCount": 200000,
    "T_sim": 1100,
    "pLogInitStd": 0.4,  # initial pLvl dispersion (cstwMPC life-cycle, SCF young households)
    "pLogInitMean": -0.5 * 0.4**2,  # Jensen correction so E[pLvl] = 1.0
    "pLvlInitCount": 25,  # discretization of newborn pLvl (default 15 is adequate but 25 is smoother)
    "kLogInitMean": np.log(0.000001),  # newborns start with ~zero assets
    "kLogInitStd": 0.0,
    # --- Solution grid (EGM) ---
    "aXtraMin": 0.0001,
    "aXtraMax": 150,
    "aXtraCount": 130,
    "aXtraNestFac": 2,  # double-exponential, matching TM grid (mFac)
    # --- Transition matrix grid ---
    # mMax=150, matching the SSJ one-asset HANK example (Auclert et al. 2021,
    # https://github.com/shade-econ/sequence-jacobian/blob/master/notebooks/hank.ipynb).
    "mMin": 1e-4,
    "mMax": 150,
    "mCount": 100,
    "mFac": 2,  # timestonest=2 → double-exponential grid, matching SSJ asset_grid
}

In [ ]:
example1 = NewKeynesianConsumerType(**Dict)
example1.solve()

In [ ]:
t0 = time.time()
# max_p_fac=10 keeps p-grid within ~exp(7.6) ≈ 2000, covering 99.99%+ of mass.
# The default (30) extends p to ~exp(23) ≈ 7.7e9, creating asset-in-levels weights
# up to 7.6e13 that amplify machine-epsilon noise into visible aggregate drift
# when iterating the transition matrix forward.
example1.define_distribution_grid(num_pointsP=110, max_p_fac=MAX_P_FAC)
t1 = time.time()
p_grid_2d = example1.dist_pGrid

example1.calc_transition_matrix()
correct_newborn_dist(example1, Dict)

t2 = time.time()
c_2d = example1.cPol_Grid
asset_2d = example1.aPol_Grid

example1.calc_ergodic_dist()
t3 = time.time()
vecDstn = example1.vec_erg_dstn

n_m_grid = len(example1.dist_mGrid)
n_p_grid = len(p_grid_2d)
n_agents = example1.AgentCount
grid_size = n_m_grid * n_p_grid
print(f"Grid: {n_m_grid} m-points × {n_p_grid} p-points = {grid_size} states")
print(f"  define_distribution_grid : {t1 - t0:6.2f}s")
print(f"  calc_transition_matrix   : {t2 - t1:6.2f}s")
print(f"  calc_ergodic_dist        : {t3 - t2:6.2f}s")
print(f"  Total                    : {t3 - t0:6.2f}s")

In [ ]:
# Compute Aggregate Consumption and Aggregate Assets (in levels = normalized × pLvl)
gridc = np.outer(c_2d, p_grid_2d)
grida = np.outer(asset_2d, p_grid_2d)

AggC = np.dot(gridc.flatten(), vecDstn)
AggA = np.dot(grida.flatten(), vecDstn)

In [ ]:
t0_new = time.time()

# The new simulator only knows variables from the YAML model file.
# Save and restore legacy track_vars around the initialize_sym() call.
_saved_track_vars = example1.track_vars[:]
example1.track_vars = ["cNrm", "aNrm", "mNrm", "pLvl"]
example1.initialize_sym()
example1.track_vars = _saved_track_vars
X = example1._simulator

n_m_2d = len(example1.dist_mGrid)
n_p_2d = len(example1.dist_pGrid)

grid_specs_2d = {
    "kNrm": {
        "min": 0.0,
        "max": float(example1.dist_mGrid[-1]),
        "N": n_m_2d,
        "order": 3,
    },
    "pLvlPrev": {
        "min": float(example1.dist_pGrid[0]),
        "max": float(example1.dist_pGrid[-1]),
        "N": n_p_2d,
        "order": 3,
    },
    "mNrm": {
        "min": 0.0,
        "max": float(example1.dist_mGrid[-1]),
        "N": n_m_2d,
        "order": 3,
    },
    "cNrm": {"min": 0.0, "max": 5.0, "N": n_m_2d, "order": 3},
    "aNrm": {
        "min": 0.0,
        "max": float(example1.dist_mGrid[-1]),
        "N": n_m_2d,
        "order": 3,
    },
}
X.make_transition_matrices(grid_specs_2d)
t1_new = time.time()

X.find_steady_state()
t2_new = time.time()

AggA_new = X.get_long_run_average("aNrm")
AggC_new = X.get_long_run_average("cNrm")

print("=== New AgentSimulator API (2D grid, no Harmenberg) ===")
print(f"  make_transition_matrices : {t1_new - t0_new:6.2f}s")
print(f"  find_steady_state        : {t2_new - t1_new:6.2f}s")
print(f"  Total                    : {t2_new - t0_new:6.2f}s")
print()
print(f"  AgentSimulator Assets = {AggA_new:.6f}")
print(f"  Legacy TM Assets      = {float(np.asarray(AggA).flat[0]):.6f}")
print(f"  AgentSimulator Cons   = {AggC_new:.6f}")
print(f"  Legacy TM Cons        = {float(np.asarray(AggC).flat[0]):.6f}")
print()
print("NOTE: Differences are expected — the two systems use different grid")
print("construction methods (uniform vs exponential spacing).  The 2D case")
print("is particularly sensitive to grid design.  The Harmenberg 1D case")
print("below provides a fairer comparison.")

In [ ]:
# [dist_new_api_market_resources] Distribution of mNrm via AgentSimulator
# The steady-state distribution over the (kNrm, pLvlPrev) arrival state space
# is stored as a flat vector.  To get the marginal over mNrm (an outcome
# variable), multiply the state distribution by the outcome projection matrix.

ss_dstn_2d = X.steady_state_dstn
mNrm_proj = X.outcome_arrays[0]["mNrm"]  # (n_states, n_mNrm_grid)
mNrm_grid_new = X.outcome_grids[0]["mNrm"]

# Marginal distribution of mNrm
mNrm_dstn_new = np.dot(ss_dstn_2d, mNrm_proj)

# Also get the kNrm arrival grid for reference
kNrm_grid_new = X.outcome_grids[0]["kNrm"]

# Compare against old marginal from erg_dstn
m_grid_old = example1.dist_mGrid
mdstn_old = example1.erg_dstn.sum(axis=1)  # marginal over p

plt.figure(figsize=(14, 6))
plt.plot(
    m_grid_old,
    mdstn_old,
    label="Legacy TM (erg_dstn marginal)",
    color=COLOR_TM,
    linewidth=2,
)
plt.plot(
    mNrm_grid_new,
    mNrm_dstn_new,
    "--",
    label="AgentSimulator (outcome projection)",
    color="tab:green",
    linewidth=2,
)
plt.ylabel("Probability Mass")
plt.xlabel("Normalized Market Resources")
plt.title("Marginal Distribution of mNrm: Legacy TM vs AgentSimulator")
plt.legend()
plt.xlim([0, 10])
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("DIAGNOSTIC: Comparing data structures")
print("=" * 60)

# Legacy system
print("\n--- Legacy TM ---")
print(f"  erg_dstn shape: {example1.erg_dstn.shape}")
print(f"  erg_dstn sum:   {example1.erg_dstn.sum():.10f}")
print(
    f"  dist_mGrid:     N={len(example1.dist_mGrid)}, [{example1.dist_mGrid[0]:.4f}, {example1.dist_mGrid[-1]:.4f}]"
)
print(
    f"  dist_pGrid:     N={len(example1.dist_pGrid)}, [{example1.dist_pGrid[0]:.4f}, {example1.dist_pGrid[-1]:.4f}]"
)
mdstn_old = example1.erg_dstn.sum(axis=1)
print(f"  mNrm marginal:  N={len(mdstn_old)}, sum={mdstn_old.sum():.10f}")
print(f"  mNrm marginal first 10: {mdstn_old[:10]}")

# New system
print("\n--- New AgentSimulator ---")
print(
    f"  steady_state_dstn: N={len(X.steady_state_dstn)}, sum={X.steady_state_dstn.sum():.10f}"
)
print(f"  outcome_arrays keys: {list(X.outcome_arrays[0].keys())}")
print(f"  outcome_grids keys:  {list(X.outcome_grids[0].keys())}")

mNrm_proj = X.outcome_arrays[0]["mNrm"]
mNrm_grid_new = X.outcome_grids[0]["mNrm"]
print(f"\n  mNrm projection matrix shape: {mNrm_proj.shape}")
print(
    f"  mNrm grid: N={len(mNrm_grid_new)}, [{mNrm_grid_new[0]:.4f}, {mNrm_grid_new[-1]:.4f}]"
)
print(
    f"  mNrm proj row sums (should be ~1): min={mNrm_proj.sum(axis=1).min():.6f}, max={mNrm_proj.sum(axis=1).max():.6f}"
)

mNrm_dstn_new = np.dot(X.steady_state_dstn, mNrm_proj)
print(f"  mNrm marginal:  N={len(mNrm_dstn_new)}, sum={mNrm_dstn_new.sum():.10f}")
print(f"  mNrm marginal first 10: {mNrm_dstn_new[:10]}")

# State space comparison
print("\n--- State space comparison ---")
state_grids_0 = X.state_grids[0]
print(f"  Number of state grid points: {len(state_grids_0)}")
if len(state_grids_0) > 0:
    first_pt = state_grids_0[0]
    last_pt = state_grids_0[-1]
    print(f"  First state point: {first_pt}")
    print(f"  Last state point:  {last_pt}")
    print(f"  State point type:  {type(first_pt)}")

# Arrival variable info
print(f"\n  Arrival variables: {X.periods[0].arrival}")
print(f"  trans_arrays: N={len(X.trans_arrays)}, shape={X.trans_arrays[0].shape}")

# Compare grids directly
print("\n--- Grid comparison ---")
print(f"  Legacy mGrid first 5:  {example1.dist_mGrid[:5]}")
print(f"  New mNrm grid first 5: {mNrm_grid_new[:5]}")
print(f"  Legacy mGrid last 5:   {example1.dist_mGrid[-5:]}")
print(f"  New mNrm grid last 5:  {mNrm_grid_new[-5:]}")

# Compute mean mNrm from both distributions
mean_m_old = np.dot(mdstn_old, example1.dist_mGrid)
mean_m_new = np.dot(mNrm_dstn_new, mNrm_grid_new)
print(f"\n  Mean mNrm (legacy): {mean_m_old:.6f}")
print(f"  Mean mNrm (new):    {mean_m_new:.6f}")

In [ ]:
# Detailed grid comparison: kNrm arrival grid vs legacy dist_mGrid
pts = X.state_grids[0]
all_pts = np.array([list(pt) if isinstance(pt, (list, tuple)) else [pt] for pt in pts])
print(f"State grid array shape: {all_pts.shape}")

if all_pts.ndim == 2 and all_pts.shape[1] == 2:
    kNrm_all = all_pts[:, 0]
    pLvlPrev_all = all_pts[:, 1]
    kNrm_arr = np.sort(np.unique(kNrm_all))
    pLvlPrev_arr = np.sort(np.unique(pLvlPrev_all))
else:
    kNrm_arr = np.sort(np.unique(all_pts.flatten()))
    pLvlPrev_arr = np.array([])

print("--- Arrival grids extracted from state_grids ---")
print(f"  kNrm unique: N={len(kNrm_arr)}, [{kNrm_arr[0]:.6f}, {kNrm_arr[-1]:.6f}]")
print(f"  kNrm first 10: {kNrm_arr[:10]}")
if len(pLvlPrev_arr) > 0:
    print(
        f"  pLvlPrev unique: N={len(pLvlPrev_arr)}, [{pLvlPrev_arr[0]:.6f}, {pLvlPrev_arr[-1]:.6f}]"
    )
    print(f"  pLvlPrev first 10: {pLvlPrev_arr[:10]}")
print()

# Check what make_exponential_grid produces
from HARK.utilities import make_exponential_grid

test_grid = make_exponential_grid(
    0.0, float(example1.dist_mGrid[-1]), len(example1.dist_mGrid), 3
)
print(
    f"  make_exponential_grid(0, {example1.dist_mGrid[-1]:.2f}, {len(example1.dist_mGrid)}, order=3):"
)
print(f"    first 10: {test_grid[:10]}")
if len(kNrm_arr) == len(test_grid):
    print(f"    Match kNrm? {np.allclose(test_grid, kNrm_arr)}")
else:
    print(f"    Different sizes: test_grid={len(test_grid)}, kNrm={len(kNrm_arr)}")
print()

# Legacy grid for comparison
from HARK.utilities import make_grid_exp_mult

legacy_grid = make_grid_exp_mult(
    example1.mMin, example1.mMax, example1.mCount, example1.mFac
)
print(
    f"  Legacy grid (make_grid_exp_mult, mMin={example1.mMin}, mMax={example1.mMax}, N={example1.mCount}, fac={example1.mFac}):"
)
print(f"    first 10: {legacy_grid[:10]}")
print()

# Key question: is the new system's kNrm grid the same as the legacy dist_mGrid?
if len(kNrm_arr) == len(example1.dist_mGrid):
    print(
        f"  Legacy dist_mGrid same as kNrm_arr? {np.allclose(example1.dist_mGrid, kNrm_arr)}"
    )
else:
    print(
        f"  Different sizes: dist_mGrid={len(example1.dist_mGrid)} vs kNrm={len(kNrm_arr)}"
    )

# Crucial: the grid_specs specify min=0, max=dist_mGrid[-1]=150, order=3
# but the legacy grid uses make_grid_exp_mult(mMin=1e-4, mMax=150, N=100, fac=2)
# These are VERY different grids
print()
print("CRITICAL: grid_specs kNrm min=0, max=150, N=100, order=3")
print(
    f"  vs legacy: min={example1.mMin}, max={example1.mMax}, N={example1.mCount}, fac={example1.mFac}"
)
print(
    "  order=3 gives MUCH denser near zero; fac=2 (double exponential) is less extreme"
)

In [ ]:
# Compare transition matrices directly
print("--- Transition matrix comparison ---")
TM_old = example1.tran_matrix
TM_new = X.trans_arrays[0]
print(f"  Legacy tran_matrix: type={type(TM_old)}, shape=", end="")
if sp.issparse(TM_old):
    print(f"{TM_old.shape} (sparse, nnz={TM_old.nnz})")
    TM_old_dense = TM_old.toarray()
else:
    print(f"{TM_old.shape}")
    TM_old_dense = np.array(TM_old)
print(f"  New trans_arrays[0]: shape={TM_new.shape}")
print()

# Row sums (should be 1 for a valid transition matrix)
old_row_sums = TM_old_dense.sum(axis=1) if TM_old_dense.ndim == 2 else None
new_row_sums = TM_new.sum(axis=1)
if old_row_sums is not None:
    print(
        f"  Legacy row sums: min={old_row_sums.min():.8f}, max={old_row_sums.max():.8f}, mean={old_row_sums.mean():.8f}"
    )
print(
    f"  New row sums:    min={new_row_sums.min():.8f}, max={new_row_sums.max():.8f}, mean={new_row_sums.mean():.8f}"
)
print()

# Col sums — for ergodic, the steady state is the left eigenvector
old_col_sums = TM_old_dense.sum(axis=0) if TM_old_dense.ndim == 2 else None
new_col_sums = TM_new.sum(axis=0)
if old_col_sums is not None:
    print(
        f"  Legacy col sums: min={old_col_sums.min():.8f}, max={old_col_sums.max():.8f}"
    )
print(f"  New col sums:    min={new_col_sums.min():.8f}, max={new_col_sums.max():.8f}")
print()

# Verify: steady_state_dstn @ TM = steady_state_dstn
residual = np.dot(X.steady_state_dstn, TM_new) - X.steady_state_dstn
print(f"  SS check ||dstn @ TM - dstn||: {np.max(np.abs(residual)):.2e}")
vec_dstn = example1.vec_erg_dstn.flatten()
old_residual = np.dot(TM_old_dense, vec_dstn) - vec_dstn
print(f"  Legacy SS check ||TM @ dstn - dstn||: {np.max(np.abs(old_residual)):.2e}")
print()
print("CRITICAL FINDING:")
print("  Legacy TM is COLUMN-stochastic: TM[to, from], cols sum to 1")
print("  New TM is ROW-stochastic: TM[from, to], rows sum to 1")
print("  Legacy SS: TM @ dstn = dstn (right eigenvector)")
print("  New SS: dstn @ TM = dstn (left eigenvector)")

In [ ]:
# Detailed distribution comparison with multiple views
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: probability mass (same as the main plot)
ax = axes[0, 0]
ax.plot(m_grid_old, mdstn_old, label="Legacy TM", color=COLOR_TM, linewidth=2)
ax.plot(
    mNrm_grid_new, mNrm_dstn_new, "--", label="New API", color="tab:green", linewidth=2
)
ax.set_title("Probability Mass (PMF)")
ax.set_xlabel("mNrm")
ax.set_xlim([0, 10])
ax.legend()

# Panel 2: CDF comparison
ax = axes[0, 1]
cdf_old = np.cumsum(mdstn_old)
cdf_new = np.cumsum(mNrm_dstn_new)
ax.plot(m_grid_old, cdf_old, label="Legacy TM", color=COLOR_TM, linewidth=2)
ax.plot(mNrm_grid_new, cdf_new, "--", label="New API", color="tab:green", linewidth=2)
ax.set_title("CDF")
ax.set_xlabel("mNrm")
ax.legend()

# Panel 3: steady-state distribution over the arrival state space
ax = axes[1, 0]
ss_dstn = X.steady_state_dstn
n_total = len(ss_dstn)
n_k = len(kNrm_arr)
n_p = len(pLvlPrev_arr) if len(pLvlPrev_arr) > 0 else 1
if n_k * n_p == n_total:
    ss_2d = ss_dstn.reshape(n_k, n_p)
    new_k_marginal = ss_2d.sum(axis=1)
    ax.plot(
        kNrm_arr,
        new_k_marginal,
        label="New API kNrm marginal (sum over p)",
        color="tab:green",
        linewidth=2,
    )
    ax.plot(
        m_grid_old,
        mdstn_old,
        "--",
        label="Legacy mNrm marginal",
        color=COLOR_TM,
        linewidth=2,
    )
    ax.set_title("Arrival state marginals (kNrm vs mNrm)")
    ax.legend()
else:
    ax.text(
        0.5, 0.5, f"n_k*n_p={n_k * n_p} != n_total={n_total}", transform=ax.transAxes
    )
ax.set_xlabel("Normalized resources / capital")

# Panel 4: log-scale comparison
ax = axes[1, 1]
ax.semilogy(
    m_grid_old,
    np.maximum(mdstn_old, 1e-20),
    label="Legacy TM",
    color=COLOR_TM,
    linewidth=2,
)
ax.semilogy(
    mNrm_grid_new,
    np.maximum(mNrm_dstn_new, 1e-20),
    "--",
    label="New API mNrm projection",
    color="tab:green",
    linewidth=2,
)
ax.set_title("Log-scale comparison")
ax.set_xlabel("mNrm")
ax.set_xlim([0, 30])
ax.legend()

plt.suptitle("Debug: Legacy TM vs New AgentSimulator distributions", fontsize=14)
plt.tight_layout()
plt.show()

# Key insight: compare kNrm marginal (arrival) vs mNrm projection (outcome)
print("=== Distribution comparison notes ===")
print("The legacy erg_dstn marginal is over mNrm (market resources).")
print("The new API steady_state_dstn is over kNrm (beginning-of-period capital).")
print("kNrm = aNrm (end-of-last-period assets), mNrm = Rfree*kNrm/PermGroFac + yNrm")
print()

# NEW: Check if the problem is in the grid or the distribution
# Compute mean of mNrm from both systems
mean_m_old = np.dot(mdstn_old, m_grid_old)
mean_m_new_proj = np.dot(mNrm_dstn_new, mNrm_grid_new)
print(f"Mean mNrm (legacy):     {mean_m_old:.6f}")
print(f"Mean mNrm (new proj):   {mean_m_new_proj:.6f}")
print()

# Check the kNrm arrival distribution directly
if n_k * n_p == n_total:
    new_k_marginal = ss_2d.sum(axis=1)
    mean_k_new = np.dot(new_k_marginal, kNrm_arr)
    print(f"Mean kNrm (new arrival): {mean_k_new:.6f}")
    print(f"Sum of kNrm marginal:    {new_k_marginal.sum():.10f}")
    print()

    # Where is the mass in each distribution?
    for pct in [0.5, 0.9, 0.95, 0.99]:
        cdf_old = np.cumsum(mdstn_old)
        idx_old = np.searchsorted(cdf_old, pct)
        val_old = m_grid_old[min(idx_old, len(m_grid_old) - 1)]

        cdf_new = np.cumsum(mNrm_dstn_new)
        idx_new = np.searchsorted(cdf_new, pct)
        val_new = mNrm_grid_new[min(idx_new, len(mNrm_grid_new) - 1)]

        cdf_k = np.cumsum(new_k_marginal)
        idx_k = np.searchsorted(cdf_k, pct)
        val_k = kNrm_arr[min(idx_k, len(kNrm_arr) - 1)]

        print(
            f"  {pct * 100:.0f}th pctl: legacy mNrm={val_old:.3f}, new mNrm proj={val_new:.3f}, kNrm arrival={val_k:.3f}"
        )